[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaytAlhikmah/hands-on-llms-for-swes/blob/main/sessions/2/notebook.ipynb)

# Session 2: Measuring Uncertainty

**Read this alongside `lesson.md`** — this notebook contains exercises referenced in the lesson.

In session 1, you saw that a language model does one thing: predict the next token. But how do you measure whether those predictions are any good? This session builds Shannon's answer from scratch — first the measure of unpredictability (entropy), then the measure of prediction quality (cross-entropy), and finally the decomposition that reveals what training actually optimizes (KL divergence).

---

## Part 0: Understanding Entropy

_Corresponds to lesson sections 0-3_

Before we measure prediction quality, we need to understand **entropy** — Shannon's measure of unpredictability. These exercises build intuition for what entropy means and why it matters.

### Setup

In [ ]:
# Install the course package
#!pip install -q git+https://github.com/BaytAlhikmah/hands-on-llms-for-swes.git#subdirectory=pkg
!pip install -e ../../pkg/
print('✓ Setup complete!')

### Exercise 1: Binary Decision Tree (Uniform Case)

_Lesson section 1: The guessing game with 8 equally likely options_

Visualize how binary search works when all outcomes are equally likely. With 8 options, you always need exactly log₂(8) = 3 questions.

In [ ]:
import matplotlib.pyplot as plt
from alhikmah_llms import Session2

# Visualize the uniform case: 8 equally likely outcomes
Session2.draw_uniform_tree()
plt.show()

print("Entropy = log₂(8) = 3.0 bits")
print("Every path from root to leaf takes exactly 3 questions.")

### Exercise 2: Binary Decision Tree (Biased Case)

_Lesson section 1: The guessing game with biased probabilities_

When outcomes aren't equally likely, you can do better! Check the most probable outcome first.

In [ ]:
# Visualize the biased case: 50% on option 1, rest split among 2-8
Session2.draw_biased_tree()
plt.show()

print("Entropy ≈ 2.4035 bits")
print("50% of the time, you're done in 1 question!")
print("Average: 0.5 × 1 + 0.5 × 3.807 = 2.4035 questions")

### Exercise 3: Visualize Probability Distributions

_Lesson section 1: How entropy relates to the "spread" of probabilities_

The more "spiky" the distribution, the lower the entropy.

In [ ]:
# Compare the distributions
Session2.plot_distributions()
plt.show()

print("Key Observation:")
print("   Uniform (flat) = maximum entropy = maximum unpredictability")
print("   Biased (spiky) = lower entropy = more predictable")

### Exercise 4: Compute Entropy for Different Distributions

_Lesson section 2: Deriving the entropy formula from first principles_

Verify that the formula H = Σ p(x) log₂(1/p(x)) matches our intuition.

In [ ]:
import math

def entropy(probs: list[float]) -> float:
    """Average number of yes/no questions (bits) to guess the outcome."""
    return sum(p * math.log2(1/p) for p in probs if p > 0)

# Test the formula
uniform_8 = [1/8] * 8
biased = [0.5] + [0.5/7] * 7
certain = [1.0] + [0.0] * 7

print(f"Uniform (all equal):     {entropy(uniform_8):.4f} bits  (= log₂(8))")
print(f"Biased (50% on one):     {entropy(biased):.4f} bits")
print(f"Certain (100% on one):   {entropy(certain):.4f} bits")
print()
print("Lower entropy = more predictable = fewer questions needed")

### Exercise 5: Implement and Test the Entropy Function

_Lesson section 2: Understanding the two equivalent forms_

The formula can be written as -Σ p log₂(p) or Σ p log₂(1/p). Why are they equivalent?

In [ ]:
# Two equivalent forms
def entropy_form1(probs: list[float]) -> float:
    """Form 1: Σ p(x) × log₂(1/p(x))"""
    return sum(p * math.log2(1/p) for p in probs if p > 0)

def entropy_form2(probs: list[float]) -> float:
    """Form 2: -Σ p(x) × log₂(p(x))"""
    return -sum(p * math.log2(p) for p in probs if p > 0)

# Verify they're the same
test_probs = [0.6, 0.2, 0.1, 0.1]
h1 = entropy_form1(test_probs)
h2 = entropy_form2(test_probs)

print(f"Form 1: {h1:.6f} bits")
print(f"Form 2: {h2:.6f} bits")
print(f"Difference: {abs(h1 - h2):.10f}")
print()
print("Why equivalent? Because log₂(1/p) = log₂(1) - log₂(p) = 0 - log₂(p) = -log₂(p)")

### Exercise 6: Compression Visualization

_Lesson section 3B: Entropy as optimal compression size_

See how entropy-based encoding saves space by assigning short codes to frequent symbols.

In [ ]:
Session2.plot_compression_example()
plt.show()

print("Key Insight:")
print("   Entropy = theoretical compression limit")
print("   Frequent symbols get short codes, rare symbols get long codes")

### Exercise 7: Surprise Curve

_Lesson section 3C: Entropy as average surprise_

See how surprise = -log₂(p) relates to probability.

In [ ]:
Session2.plot_surprise_curve()
plt.show()

print("Key Insight:")
print("   High probability → low surprise (you expected it)")
print("   Low probability → high surprise (you didn't see that coming!)")
print("   Entropy = expected surprise across all possible outcomes")

### Exercise 8: Explore Entropy Across Distributions

_Lesson section 3: Interactive visualizations_

#### 8a. Binary Entropy Function

The most fundamental curve in information theory — entropy of a coin flip with probability p.

In [ ]:
Session2.plot_binary_entropy()
plt.show()

print("Key Observation:")
print("   Maximum entropy at p=0.5 (fair coin flip)")
print("   Zero entropy at p=0 or p=1 (certain outcome)")
print("   Symmetric curve — this is the foundation of all entropy calculations!")

#### 8b. Ternary Entropy (3-outcome distributions)

Shows entropy for all possible 3-outcome probability distributions.

In [ ]:
Session2.plot_ternary_entropy_3d_plotly().show()

print("Key Observation:")
print("   Click and drag to rotate the 3D surface!")
print("   Peak at center (uniform 1/3, 1/3, 1/3) = maximum entropy ≈ 1.585 bits")
print("   Valleys at corners (certain outcome) = zero entropy")

#### 8c. Mathematical Properties of Entropy

Now that you've seen entropy in action, let's formalize its key mathematical properties. These properties uniquely characterize the entropy function and explain why it's the "right" measure of uncertainty.

**Property 1: Non-negativity**
```
H(X) ≥ 0
```
Entropy is always non-negative. It equals zero only when one outcome has probability 1 (complete certainty).

**Property 2: Maximum Entropy**
```
H(X) ≤ log₂(n)
```
For n possible outcomes, entropy is maximized when all outcomes are equally likely (uniform distribution). Maximum = log₂(n) bits.

**Property 3: Continuity**
H is a continuous function of the probability distribution. Small changes in probabilities lead to small changes in entropy.

**Property 4: Symmetry**
H is invariant under permutation of outcomes. The order doesn't matter, only the probabilities.

**Property 5: Additivity (for independent events)**
```
H(X,Y) = H(X) + H(Y)  [if X and Y are independent]
```
The joint entropy of two independent random variables equals the sum of their individual entropies.

**Property 6: Chain Rule**
```
H(X,Y) = H(X) + H(Y|X)
```
Joint entropy equals the entropy of X plus the conditional entropy of Y given X.

**Property 7: Conditioning Reduces Entropy**
```
H(Y|X) ≤ H(Y)
```
Knowing X can only reduce (or leave unchanged) uncertainty about Y. Information never increases uncertainty. Equality holds when X and Y are independent.

These properties aren't just mathematical curiosities — they capture fundamental facts about information and uncertainty.

In [ ]:
# Demonstrate the properties with examples

# Property 1: Non-negativity
print("Property 1: Non-negativity")
print("-" * 50)
certain = [1.0, 0.0, 0.0]
uniform = [1/3, 1/3, 1/3]
biased = [0.7, 0.2, 0.1]

print(f"  Certain outcome:  H = {entropy(certain):.4f} bits  (minimum)")
print(f"  Uniform:          H = {entropy(uniform):.4f} bits")
print(f"  Biased:           H = {entropy(biased):.4f} bits")
print(f"  All ≥ 0 ✓\n")

# Property 2: Maximum Entropy
print("Property 2: Maximum Entropy")
print("-" * 50)
n = 8
uniform_8 = [1/8] * 8
max_entropy = math.log2(n)
actual_entropy = entropy(uniform_8)
print(f"  For n={n} outcomes:")
print(f"  Maximum possible:  log₂({n}) = {max_entropy:.4f} bits")
print(f"  Uniform achieves:  H = {actual_entropy:.4f} bits")
print(f"  Any other distribution will have H < {max_entropy:.4f}")

# Test with non-uniform
non_uniform = [0.5, 0.1, 0.1, 0.1, 0.1, 0.05, 0.03, 0.02]
print(f"  Non-uniform:       H = {entropy(non_uniform):.4f} bits < {max_entropy:.4f} ✓\n")

# Property 4: Symmetry
print("Property 4: Symmetry")
print("-" * 50)
dist1 = [0.5, 0.3, 0.2]
dist2 = [0.2, 0.5, 0.3]  # Same probabilities, different order
dist3 = [0.3, 0.2, 0.5]  # Another permutation
print(f"  [0.5, 0.3, 0.2] → H = {entropy(dist1):.6f} bits")
print(f"  [0.2, 0.5, 0.3] → H = {entropy(dist2):.6f} bits")
print(f"  [0.3, 0.2, 0.5] → H = {entropy(dist3):.6f} bits")
print(f"  Order doesn't matter, only the probabilities ✓\n")

# Property 5: Additivity (independent events)
print("Property 5: Additivity for Independent Events")
print("-" * 50)
# Two independent coin flips
coin1 = [0.5, 0.5]  # Fair coin
coin2 = [0.6, 0.4]  # Biased coin

H_coin1 = entropy(coin1)
H_coin2 = entropy(coin2)

# Joint distribution for independent events: P(X,Y) = P(X) * P(Y)
joint = [coin1[i] * coin2[j] for i in range(2) for j in range(2)]
H_joint = entropy(joint)

print(f"  Coin 1:            H = {H_coin1:.4f} bits")
print(f"  Coin 2:            H = {H_coin2:.4f} bits")
print(f"  Sum:               H₁ + H₂ = {H_coin1 + H_coin2:.4f} bits")
print(f"  Joint (independent): H(X,Y) = {H_joint:.4f} bits")
print(f"  Difference: {abs(H_joint - (H_coin1 + H_coin2)):.6f} (should be ~0) ✓\n")

# Property 7: Conditioning Reduces Entropy
print("Property 7: Conditioning Reduces Entropy")
print("-" * 50)
print("  This is the key property for prediction!")
print("  Knowing context can only reduce uncertainty about what comes next.")
print("  We'll apply this in Session 3 with real sequence data:\n")
print("  - H(next move) without knowing previous = higher entropy")
print("  - H(next move | previous move) = lower entropy")
print("  - The reduction tells us how much context helps predict the next outcome\n")

print("Key Takeaway:")
print("   These properties explain why entropy is the 'right' way to measure uncertainty.")
print("   Any other measure that satisfies these axioms must be equivalent to Shannon entropy.")

---

## Part 1: Cross-Entropy — Formalizing "Loss"

_Corresponds to lesson sections 4-6_

In Part 0, we called entropy "average surprise" — how surprised you are on average when outcomes are drawn from distribution P, and you know P. In machine learning, we have a different situation: the true distribution is P, but our model predicts a *different* distribution Q. How surprised is the model?

This is called **cross-entropy**, and it's the loss function behind every language model.

### Exercise 9: The Core Intuition

_Lesson section 4: What is cross-entropy?_

Cross-entropy H(P,Q) measures how well a predicted distribution Q matches the true distribution P.

**Example:** After 'q', what character comes next?
- True distribution P: 'u' appears 70% of the time, 'a' 20%, 'e' 10%
- Bad model Q: predicts uniformly (33% each)
- Good model Q: predicts close to true (65%, 25%, 10%)

**Lower cross-entropy = better model**

In [ ]:
import numpy as np
from alhikmah_llms import Session3

# Visualize: true distribution vs two models
Session3.plot_cross_entropy_intuition()
plt.show()

print("Key Insight:")
print("   The model with predictions closer to the true distribution")
print("   has LOWER cross-entropy = BETTER performance!")

### Exercise 10: Where Does the Loss Come From?

_Lesson section 5: Loss breakdown_

Cross-entropy is computed as:

```
H(P,Q) = -∑ P(x) log Q(x)
```

Each outcome contributes `-P(x) log Q(x)` to the total loss.

**Question:** Which outcomes matter most?

In [ ]:
# Breakdown: see which outcomes contribute most
Session3.plot_surprise_breakdown()
plt.show()

print("\nKey Observation:")
print("   Outcomes with HIGH TRUE PROBABILITY (large P) contribute most!")
print("   Getting common cases right matters more than rare ones.")

### Exercise 11: The Mathematical Formula

_Lesson section 5: Computing cross-entropy_

Let's implement cross-entropy from scratch and verify it matches our intuition.

In [ ]:
def cross_entropy(P: list[float], Q: list[float]) -> float:
    """Compute H(P,Q) = -∑ P(x) log Q(x)
    
    Args:
        P: True probability distribution
        Q: Predicted probability distribution
    
    Returns:
        Cross-entropy in nats (natural log units)
    """
    return -sum(p * np.log(q) if q > 0 else 0 for p, q in zip(P, Q))


# Test cases
P_true = [0.7, 0.2, 0.1]

# Perfect prediction
Q_perfect = [0.7, 0.2, 0.1]
print(f"Perfect model: H(P,Q) = {cross_entropy(P_true, Q_perfect):.4f}")

# Good prediction
Q_good = [0.65, 0.25, 0.1]
print(f"Good model:    H(P,Q) = {cross_entropy(P_true, Q_good):.4f}")

# Bad prediction (uniform)
Q_bad = [0.33, 0.33, 0.34]
print(f"Bad model:     H(P,Q) = {cross_entropy(P_true, Q_bad):.4f}")

# Terrible prediction (backwards!)
Q_terrible = [0.1, 0.2, 0.7]
print(f"Terrible model: H(P,Q) = {cross_entropy(P_true, Q_terrible):.4f}")

print("\nNotice: Loss increases as predicted distribution gets worse!")

### Exercise 12: Bits vs Nats

_Lesson section 6: Units of information_

**Important:** In Part 0, we used **log₂** and measured entropy in **bits**. In machine learning (and from now on), we use **natural log (ln)** and measure in **nats**.

Why?
- PyTorch's `nn.CrossEntropyLoss()` uses natural log
- Derivatives are cleaner: d/dx(ln x) = 1/x
- Standard in ML papers and frameworks

**Conversion:** `bits = nats / ln(2) ≈ nats * 1.443`

In [ ]:
# Compare bits vs nats
P = [0.7, 0.2, 0.1]
Q = [0.65, 0.25, 0.1]

# In nats (natural log)
ce_nats = cross_entropy(P, Q)

# In bits (log2)
ce_bits = -sum(p * math.log2(q) if q > 0 else 0 for p, q in zip(P, Q))

print(f"Cross-entropy in NATS: {ce_nats:.4f}")
print(f"Cross-entropy in BITS: {ce_bits:.4f}")
print(f"Ratio: {ce_bits / ce_nats:.4f} ≈ 1 / ln(2) ≈ 1.443")

print("\nFrom now on, we use NATS (natural log) to match ML conventions.")

---

## Part 2: The Key Decomposition — Entropy + KL Divergence

_Corresponds to lesson sections 7-8_

Here's the most important equation in this session:

```
H(P,Q) = H(P) + KL(P||Q)
```

Where:
- **H(P,Q)** = cross-entropy (what we minimize in training)
- **H(P)** = entropy of true distribution (constant, can't change)
- **KL(P||Q)** = KL divergence (measures how Q differs from P)

### Exercise 13: Visualizing the Decomposition

_Lesson section 7: The decomposition_

**Key insight:** When training a model, we can't change H(P) (it's a property of the data). We're really just minimizing KL(P||Q)!

In [ ]:
Session3.plot_entropy_decomposition()
plt.show()

print("\nCritical Insight:")
print("   Training doesn't reduce H(P) — that's fixed by the data!")
print("   Training minimizes KL(P||Q) — making Q match P.")
print("   Minimum loss = H(P) when Q = P (KL = 0)")

### Exercise 14: Computing KL Divergence

_Lesson section 8: What is KL divergence?_

KL divergence (Kullback-Leibler divergence) measures how one distribution differs from another:

```
KL(P||Q) = ∑ P(x) log(P(x) / Q(x))
         = ∑ P(x) log P(x) - ∑ P(x) log Q(x)
         = -H(P) + H(P,Q)
```

Properties:
- Always non-negative: KL(P||Q) ≥ 0
- Zero only when P = Q
- **NOT symmetric:** KL(P||Q) ≠ KL(Q||P)

In [ ]:
def entropy_nats(P: list[float]) -> float:
    """Compute H(P) = -∑ P(x) log P(x)"""
    return -sum(p * np.log(p) for p in P if p > 0)

def kl_divergence(P: list[float], Q: list[float]) -> float:
    """Compute KL(P||Q) = ∑ P(x) log(P(x) / Q(x))"""
    return sum(p * np.log(p / q) if p > 0 and q > 0 else 0 
               for p, q in zip(P, Q))

# Verify the decomposition
P = [0.7, 0.2, 0.1]
Q = [0.5, 0.3, 0.2]

h_p = entropy_nats(P)
h_pq = cross_entropy(P, Q)
kl_pq = kl_divergence(P, Q)

print("Verify H(P,Q) = H(P) + KL(P||Q):")
print(f"  H(P)      = {h_p:.6f}")
print(f"  KL(P||Q)  = {kl_pq:.6f}")
print(f"  H(P) + KL = {h_p + kl_pq:.6f}")
print(f"  H(P,Q)    = {h_pq:.6f}")
print(f"  Match? {abs(h_pq - (h_p + kl_pq)) < 1e-10}")

### Exercise 15: KL Divergence is NOT Symmetric

_Lesson section 8: Asymmetry matters_

This is important! KL(P||Q) ≠ KL(Q||P), and the difference matters for how models learn.

- **Forward KL:** KL(P||Q) — penalizes heavily when Q assigns low probability where P is high ("mode-seeking")
- **Reverse KL:** KL(Q||P) — penalizes heavily when Q assigns high probability where P is low ("mode-covering")

In [ ]:
Session3.plot_kl_asymmetry()
plt.show()

print("\nKey Point:")
print("   In ML, we minimize KL(P||Q) where P = data, Q = model")
print("   This is 'forward KL' — model must cover all modes of data!")

In [ ]:
# Numerical example
P = [0.8, 0.15, 0.05]  # peaked distribution
Q = [0.4, 0.3, 0.3]     # flat distribution

kl_pq = kl_divergence(P, Q)
kl_qp = kl_divergence(Q, P)

print(f"KL(P||Q) = {kl_pq:.4f}")
print(f"KL(Q||P) = {kl_qp:.4f}")
print(f"Difference: {abs(kl_pq - kl_qp):.4f}")
print("\nThey're NOT equal!")

---

## Part 3: Training = Minimizing Cross-Entropy

_Corresponds to lesson sections 9-10_

Now we understand what "training" really means: **adjust Q to minimize H(P,Q)**.

### Exercise 16: Watch a Model Converge

_Lesson section 9: Training visualization_

As training progresses, the predicted distribution Q moves closer to the true distribution P, and cross-entropy decreases.

In [ ]:
Session3.plot_training_trajectory()
plt.show()

print("\nWhat's Happening:")
print("   The blue bars (model) gradually match the green bars (truth)")
print("   As they align, cross-entropy decreases toward H(P)")
print("   This IS what happens inside neural network training!")

### Exercise 17: The Loss Surface

_Lesson section 10: Loss landscape_

For a 3-outcome distribution, we can visualize the entire loss surface as we vary Q's parameters.

**Key observation:** There's a unique minimum at Q = P.

In [ ]:
Session3.plot_cross_entropy_surface(
    true_probs=[0.7, 0.2, 0.1]
)
plt.show()

print("\nObservations:")
print("   - Red star (Q = P) is the global minimum")
print("   - Loss increases as Q moves away from P")
print("   - Gradient descent would roll downhill to the minimum")
print("   - (In neural nets, we optimize weights, not Q directly!)")

---

## Part 4: Cross-Entropy in Practice

_Corresponds to lesson sections 11-12_

Let's connect this to real prediction problems. After 'q', what character comes next?

### Exercise 18: Real Bigram Example

_Lesson section 11: Bigram cross-entropy_

Watch cross-entropy decrease as a bigram model learns.

In [ ]:
Session3.plot_bigram_cross_entropy_example()
plt.show()

print("\nWhat You're Seeing:")
print("   - Training data: 'qu' appears 850 times, others are rare")
print("   - Early training: model is nearly uniform (bad!)")
print("   - Late training: model learns to predict 'u' after 'q' (good!)")
print("   - Cross-entropy drops from ~1.6 to ~0.06 nats")

### Exercise 19: Loss Curves

_Lesson section 12: Monitoring training_

In practice, we plot loss over training epochs to monitor progress.

In [ ]:
# Generate example loss curves
Session3.plot_loss_curve()
plt.show()

print("\nWhat to Watch For:")
print("   - Training loss should decrease smoothly")
print("   - Validation loss should track training loss")
print("   - If val >> train → overfitting")
print("   - If both plateau → model converged (or stuck in local minimum)")

### Exercise 20: Custom Cross-Entropy Examples

_Lesson section 13: Experiment yourself_

Try different distributions and see how cross-entropy changes.

In [ ]:
# Example 1: Very confident but wrong
P = [0.9, 0.08, 0.02]
Q = [0.1, 0.1, 0.8]  # model is confident but backwards!

print("Example 1: Confident but wrong")
print(f"  True P = {P}")
print(f"  Model Q = {Q}")
print(f"  Cross-entropy = {cross_entropy(P, Q):.4f}")
print(f"  (Very high! Model is confidently wrong.)")
print()

# Example 2: Uncertain but safe
Q_uniform = [0.33, 0.33, 0.34]
print("Example 2: Uniform (uncertain)")
print(f"  Model Q = {Q_uniform}")
print(f"  Cross-entropy = {cross_entropy(P, Q_uniform):.4f}")
print(f"  (Lower than Example 1! Being uncertain is better than being wrong.)")
print()

# Your turn: create more examples!
# What happens when Q is very close to P?
# What happens when Q completely misses a high-probability outcome?

### Exercise 21: Visualize Your Own Examples

_Lesson section 13: Build intuition_

Use the visualization functions with your own distributions.

In [ ]:
# Create your own example
my_true = [0.6, 0.3, 0.1]
my_pred_bad = [0.2, 0.4, 0.4]
my_pred_good = [0.58, 0.32, 0.1]

Session3.plot_cross_entropy_intuition(
    true_probs=my_true,
    predicted_bad=my_pred_bad,
    predicted_good=my_pred_good,
    labels=['A', 'B', 'C']
)
plt.show()

---

## Summary and What's Next

### What We Learned

1. **Entropy** H(P) measures unpredictability — three equivalent views: average yes/no questions, optimal compression size, average surprise
2. **Cross-entropy** H(P,Q) measures how well predicted distribution Q matches true distribution P
3. **Decomposition**: H(P,Q) = H(P) + KL(P||Q)
   - H(P) is constant (property of data)
   - Training minimizes KL(P||Q)
4. **KL divergence** is NOT symmetric: KL(P||Q) ≠ KL(Q||P)
5. **Training** = adjusting model parameters to minimize cross-entropy
6. We use **natural log (nats)** in ML, not log₂ (bits)

### What's Next (Session 3)

Now that you have the theoretical foundation, Session 3 applies these ideas to real data:
- **Rock-Paper-Scissors:** Build a transition matrix, measure its entropy, and exploit patterns to win
- **Character bigrams:** Scale up to predicting the next character in names (28×28 matrix)
- **The V² wall:** Why transition matrices don't scale to real vocabularies

---

## Exercises for Practice

1. **Derive the decomposition**: Prove that H(P,Q) = H(P) + KL(P||Q) using the definitions
2. **Find worst case**: For P = [0.7, 0.2, 0.1], what Q maximizes cross-entropy?
3. **Asymmetry exploration**: Create distributions where KL(P||Q) and KL(Q||P) differ dramatically
4. **Temperature**: If Q = softmax(logits / T), how does temperature affect cross-entropy?